In [ ]:


import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
pd.set_option('display.max_columns', None)

PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Products' / 'Chamber Study Missions' / '2026'
PATH_I = Path(r'I:\Projects\Josh\Regional Monitoring')

def re_remove_post(x, exp = ' '):
    try:
        return x.split(exp, 1)[0]
    except Exception as e:
        e
        return x



In [ ]:

EXPORT=False

path_in = PATH_SP / 'MnR_outputs'

list_files = [file for file in path_in.iterdir() if file.is_file()]
list_files_bls = [file for file in path_in.iterdir() if 'BLS' in str(file.stem)]

print(list_files_bls)

indicators_jobs  = ['Jobs_1', 'Jobs_2', 'Jobs_3', 'Labor_2']



In [ ]:


for file in list_files_bls:

    indicator = re_remove_post(file.stem)
    print(f'{file.stem}.xlsx')

    if indicator in ['Jobs_1', 'Jobs_2', 'Labor_2']:
        df = pd.read_excel(file, sheet_name='MSA')
        df['MSA ID'] = df['MSA ID'].astype(str)
        df = df.rename(columns={'Geography':'MSA', 'Percentage':'Percent'})

        if 'Notes' in df.columns:
            df = df.drop('Notes', axis=1)

        if indicator == 'Labor_2':
            file_weights = path_in / 'Jobs_1 MSA BLS SM.xlsx'
            df_weights = pd.read_excel(file_weights, sheet_name='MSA')
            df_weights = df_weights[df_weights['Sector'] == 'All']
            # df_weights = df_weights.rename(columns={'MSA ID':'MSA_ID'})

            df_weights = df_weights[['MSA ID', 'date_', 'Total Jobs']]
            df_weights['MSA ID'] = df_weights['MSA ID'].astype(str)
            df = df.merge(df_weights, on=['MSA ID', 'date_'], how='left')

        df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
        df.loc[df['MSA'] == 'SACOG', 'MSA ID'] = '40900, 49700'
        df.loc[df['MSA'].str.contains('Detroit|Monroe|Ann Arbor'), 'MSA'] = 'SEMCOG'
        df.loc[df['MSA'] == 'SEMCOG', 'MSA ID'] = '11460, 19820, 33780'

        if indicator == 'Labor_2':
            # df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            def wm(x):
                return np.average(x, weights = df.loc[x.index, 'Total Jobs'])
            df = df.groupby(['MSA ID', 'MSA', 'date_'], as_index=False).agg(Percent=('Unemployment Rate', wm))
        if indicator in ['Jobs_1', 'Jobs_2']:
            df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            df = df.groupby(['MSA ID', 'MSA', 'date_', 'Sector'], as_index=False).agg(jobs=('Total Jobs', 'sum'), Percent=('Percent', wm))

        if indicator in ['Jobs_1', 'Jobs_2']:
            df = df.rename(columns = {'jobs':'Total Jobs'})
            df['Total Jobs'] = df['Total Jobs'].replace(3, 0)
        if indicator == 'Labor_2':
            df = df.rename(columns = {'Percent':'Unemployment Rate'})

        df = df.sort_values(['MSA ID', 'date_'], ascending=[True, False])
        df = df.reset_index(drop=True)

        if indicator != 'Labor_2':
            df.loc[df['Total Jobs'] == 0, 'Notes'] = 'No data collected for this specific sector'

        display(df.head(2))


    if indicator == 'Jobs_3':

        sheets = ['Government and Private', 'Goods and Services']

        for sheet_name in sheets:

            print(sheet_name)

            df = pd.read_excel(file, sheet_name=sheet_name)
            df = df.rename(columns={'Percentage':'Percent'})
            df['MSA ID'] = df['MSA ID'].astype(str)

            df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
            df.loc[df['MSA'] == 'SACOG', 'MSA ID'] = '40900, 49700'
            df.loc[df['MSA'].str.contains('Detroit|Monroe|Ann Arbor'), 'MSA'] = 'SEMCOG'
            df.loc[df['MSA'] == 'SEMCOG', 'MSA ID'] = '11460, 19820, 33780'

            df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            df = df.groupby(['MSA ID', 'MSA', 'date_', 'Sector'], as_index=False).agg(jobs=('Total Jobs', 'sum'), Percent=('Percent', wm))

            df = df.rename(columns = {'jobs':'Total Jobs'})
            df['Total Jobs'] = df['Total Jobs'].replace(3, 0)

            df = df.sort_values(['MSA ID', 'date_'], ascending=[True, False])
            df = df.reset_index(drop=True)

            display(df.head(2))
            

    if EXPORT:
        file_out = PATH_SP /'Indicators'/ f'{file.stem}_prt_SEMCOG.xlsx'
        with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df.to_excel(writer, sheet_name='MSA', index=False)

